In [0]:
dbutils.widgets.text(name="env", defaultValue="", label="Enter the environment value")
env = dbutils.widgets.get("env")

In [0]:
%run "./commons"

In [0]:
bronze_path

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import StructField, StructType, StringType

schema = StructType([
    StructField("PatientID",StringType()),
    StructField("FirstName", StringType()),
    StructField("LastName",StringType()),
    StructField("MiddleName",StringType()),
    StructField("SSN",StringType()),
    StructField("PhoneNumber",StringType()),
    StructField("Gender",StringType()),
    StructField("DOB",StringType()),
    StructField("Address",StringType()),
    StructField("ModifiedDate",StringType()),
    StructField("Extract_Time", StringType()),
    StructField("filename", StringType())
])

def read_bronze_data(environment):
    print("Reading bronze table data")
    df_bronze = spark.readStream.option("ignoreDeletes","true").schema(schema).table(f"{environment}_catalog.bronze.raw_patient")
    return df_bronze

def create_Transformed_time(df):
    df = df.withColumn("transformed_time", current_timestamp())
    return df







In [0]:
def write_To_Silver(df):
    print("Starting to write the data into Silver schema after transformation")
    write_Stream = df.writeStream.format("delta").option("checkpointLocation",checkpoint_path + "/silver_load/").queryName("write_To_Silver").trigger(availableNow=True).outputMode("append").toTable(f"`{env}_catalog`.`silver`.`patient`")
    
    write_Stream.awaitTermination()
    print("silver table load is completed")



In [0]:
#Main execution flow:
df_bronze = read_bronze_data(env)
print("read from bronze")

df_bronze_dedup = remove_duplicates(df_bronze)
print("deduplication completed")

Allcolumns=df_bronze_dedup.schema.names
df_null_handled = handle_nulls(df_bronze_dedup,Allcolumns)
print("Null handling completed")

df_final = create_Transformed_time(df_null_handled)
print("transformation completed")

write_To_Silver(df_final)
print("finally wrote the silver table from bronze")